# st3d Demo

End-to-end walkthrough: generate synthetic serial 2D sections, preprocess,
train the model, reconstruct a 3D volume, and visualise the results.

**No real data required** -- this notebook creates synthetic tissue sections
with spatially-varying gene expression that changes smoothly along the z-axis.

## 0. Setup

In [ ]:
import sys, os
# Add the parent directory so st3d is importable when running from notebooks/
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import numpy as np
import anndata as ad
import pandas as pd
import torch
import scanpy as sc

# Seed everything for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print(f'PyTorch {torch.__version__}')
print(f'Device: {"cuda" if torch.cuda.is_available() else "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu"}')

## 1. Generate Synthetic Serial Sections

We create a simple synthetic tissue:
- **5 serial sections** at z = 0.0, 0.1, 0.2, 0.3, 0.4
- Each section has ~200 spots arranged on a noisy grid
- 500 genes, with expression that varies spatially (xy) and along depth (z)

In [ ]:
def make_synthetic_section(z_depth: float, n_spots: int = 200, n_genes: int = 500) -> ad.AnnData:
    """Generate a single synthetic ST section."""
    # Noisy grid coordinates in [0, 10] x [0, 8]
    grid_x = np.linspace(0.5, 9.5, int(np.sqrt(n_spots)))
    grid_y = np.linspace(0.5, 7.5, int(np.sqrt(n_spots)))
    xx, yy = np.meshgrid(grid_x, grid_y)
    xy = np.column_stack([xx.ravel(), yy.ravel()]).astype(np.float32)
    xy += np.random.randn(*xy.shape).astype(np.float32) * 0.15  # jitter
    n = xy.shape[0]

    # Gene expression: combination of spatial patterns that shift with z
    # Pattern 1: radial gradient from centre (shifts right with z)
    cx, cy = 5.0 + z_depth * 3, 4.0
    r = np.sqrt((xy[:, 0] - cx) ** 2 + (xy[:, 1] - cy) ** 2)
    pattern1 = np.exp(-r / 3.0)

    # Pattern 2: horizontal stripe that moves down with z
    stripe_center = 3.0 + z_depth * 4
    pattern2 = np.exp(-((xy[:, 1] - stripe_center) ** 2) / 2.0)

    # Build expression matrix: each gene is a noisy mix of patterns
    gene_weights1 = np.random.randn(n_genes).astype(np.float32) * 0.5
    gene_weights2 = np.random.randn(n_genes).astype(np.float32) * 0.5
    expr = (
        np.outer(pattern1, gene_weights1)
        + np.outer(pattern2, gene_weights2)
        + np.random.randn(n, n_genes).astype(np.float32) * 0.1
    )
    # Make non-negative (counts-like)
    expr = np.abs(expr) * 10
    expr = expr.astype(np.float32)

    gene_names = [f'Gene_{i}' for i in range(n_genes)]
    adata = ad.AnnData(
        X=expr,
        obs=pd.DataFrame({'section': z_depth}, index=[f's{z_depth:.1f}_c{j}' for j in range(n)]),
        var=pd.DataFrame(index=gene_names),
    )
    adata.obsm['spatial'] = xy
    return adata


# Generate 5 sections
z_coords = [0.0, 0.1, 0.2, 0.3, 0.4]
sections = [make_synthetic_section(z, n_spots=196, n_genes=500) for z in z_coords]

for i, s in enumerate(sections):
    print(f'Section {i}: z={z_coords[i]:.1f}, {s.n_obs} spots, {s.n_vars} genes')

## 2. Preprocess Sections

The `preprocess_sections` function handles everything: log1p, HVG selection,
PCA, coordinate normalisation. Returns tensors ready for the model.

In [ ]:
from st3d.data import preprocess_sections

tensors, norm_params = preprocess_sections(
    sections,
    z_coords=z_coords,
    n_hvgs=200,   # fewer HVGs for this small demo
    n_pcs=30,     # fewer PCs for speed
)

for i, t in enumerate(tensors):
    print(f'Section {i}: tensor shape {t.shape}  (N, 3 + {norm_params.n_pcs} PCs)')

## 3. Create and Train the Model

We use a small model configuration suitable for this demo. On real data you
would use the defaults (hidden_dim=64, n_attn_layers=3, etc.).

In [ ]:
from st3d.config import ST3DConfig
from st3d.model import ST3DModel
from st3d.training import Trainer

config = ST3DConfig(
    n_pcs=30,
    hidden_dim=32,
    n_heads=2,
    n_attn_layers=2,
    k_neighbors=8,
    dz=0.02,
    bidirectional=True,
    lr=5e-4,
    curriculum_start_epoch=0,
    curriculum_end_epoch=30,
    sinkhorn_iters=20,   # fewer iterations for speed
)

model = ST3DModel(config)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
trainer = Trainer(model, config, device='auto', checkpoint_dir='demo_ckpt')
print(f'Training on device: {trainer.device}')

history = trainer.fit(tensors, epochs=50, checkpoint_every=25)

### Training Curves

In [ ]:
from st3d.visualization import plot_training_history

plot_training_history(history)

## 4. Reconstruct 3D Volume

Interpolate between the observed sections to produce a dense 3D volume.

In [ ]:
from st3d.inference import reconstruct_volume

volume = reconstruct_volume(
    model,
    tensors,
    norm_params,
    dz=0.02,
    device='auto',
    inverse_pca=True,
)

print(f'Reconstructed volume: {volume.n_obs} points, {volume.n_vars} genes')
print(f'Z range: {volume.obs["z"].min():.3f} to {volume.obs["z"].max():.3f}')
print(f'Observed points: {sum(volume.obs["is_observed"])}, Interpolated: {sum(~volume.obs["is_observed"].astype(bool))}')

## 5. Visualise

### 5a. 3D scatter plot coloured by a gene

In [ ]:
from st3d.visualization import plot_sections_3d

fig = plot_sections_3d(volume, color_by='Gene_0', point_size=2, opacity=0.5)
fig.show()

### 5b. 3D scatter coloured by observed vs interpolated

In [ ]:
fig = plot_sections_3d(volume, color_by='is_observed', point_size=2, opacity=0.5,
                       title='Observed (1) vs Interpolated (0)')
fig.show()

### 5c. Section-by-section interpolation comparison

In [ ]:
from st3d.visualization import plot_interpolation

fig = plot_interpolation(volume, gene='Gene_0', n_sections=5)
fig.show()

## 6. Save and Load

Save the reconstructed volume as an h5ad and the model checkpoint.

In [ ]:
# Save volume
volume.write('demo_volume.h5ad')
print('Volume saved to demo_volume.h5ad')

# Load back
vol_loaded = ad.read_h5ad('demo_volume.h5ad')
print(f'Loaded: {vol_loaded.n_obs} points, {vol_loaded.n_vars} genes')

In [ ]:
# Save and reload model
model.save('demo_model.pt')
model_loaded = ST3DModel.load('demo_model.pt')
print(f'Model reloaded: {sum(p.numel() for p in model_loaded.parameters()):,} parameters')

---

**Done!** This demo showed the full pipeline:

1. Generate / load serial 2D ST sections (AnnData)
2. Preprocess: log1p, HVG, PCA, coordinate normalisation
3. Train with single-loop curriculum scheduling
4. Reconstruct dense 3D volume via bidirectional Euler integration
5. Visualise in 3D and per-section
6. Save / load results

For real data, replace `make_synthetic_section` with `scanpy.read_h5ad`
and adjust `ST3DConfig` parameters (e.g. larger `hidden_dim`, more epochs).